# Module 1: Data Ingestion & Preprocessing
This notebook correctly merges the MPLADS datasets by extracting the unique Project ID instead of concatenating them.

In [ ]:
import pandas as pd
import numpy as np
import re
import os


In [ ]:
# Load datasets
data_dir = 'data'
rec_df = pd.read_csv(os.path.join(data_dir, 'Works Recommended.csv'))
sanc_df = pd.read_csv(os.path.join(data_dir, 'Works Sanctioned.csv'))
exp_df = pd.read_csv(os.path.join(data_dir, 'Expenditure on Completed and On-going Works as on Date.csv'))
comp_df = pd.read_csv(os.path.join(data_dir, 'Works Completed.csv'))

print(f"Recommended: {len(rec_df)}")
print(f"Sanctioned: {len(sanc_df)}")
print(f"Expenditure: {len(exp_df)}")
print(f"Completed: {len(comp_df)}")

In [ ]:
# Function to extract the 6-digit project ID from the Work string
def extract_id(text):
    if pd.isna(text):
        return np.nan
    # Look for a 5 or 6 digit number that comes after a slash and before a hyphen
    match = re.search(r'/(\\d{5,7})-', str(text))
    if match:
        return match.group(1)
    # If the text itself is just a number (like in Expenditure)
    if str(text).strip().isdigit():
        return str(text).strip()
    return np.nan

In [ ]:
# Apply the extraction function to create a unified primary key
rec_df['Project_ID'] = rec_df['WORK'].apply(extract_id)
sanc_df['Project_ID'] = sanc_df['Work'].apply(extract_id)
exp_df['Project_ID'] = exp_df['Work ID'].apply(extract_id)
comp_df['Project_ID'] = comp_df['Work'].apply(extract_id)

print(f"Valid IDs extracted - Rec: {rec_df['Project_ID'].notna().sum()}, Sanc: {sanc_df['Project_ID'].notna().sum()}, Exp: {exp_df['Project_ID'].notna().sum()}, Comp: {comp_df['Project_ID'].notna().sum()}")

In [ ]:
# Clean and normalize text columns
for df in [rec_df, sanc_df]:
    if 'Work description' in df.columns:
        df['Work description'] = df['Work description'].str.lower().str.strip()
    if 'Work category' in df.columns:
        df['Work category'] = df['Work category'].str.lower().str.strip()

In [ ]:
# Parse dates
date_cols = ['Recommended date', 'Sanction Date', 'Expenditure Date', 'Completion Date']
for df in [rec_df, sanc_df, exp_df, comp_df]:
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
# Drop unnecessary columns before merging to avoid duplication
# We will use the Sanctioned DF as the anchor for description, so we just take amounts from others
rec_subset = rec_df[['Project_ID', 'Work category', 'Hon\'ble Members of Parliament', 'Constituency', 'State', 'Recommended date', 'RECOMMENDED AMOUNT   ( ₹ )']].drop_duplicates(subset=['Project_ID'])
sanc_subset = sanc_df[['Project_ID', 'Sanction Date', 'Sanction Amount ( ₹ )', 'Work Status']].drop_duplicates(subset=['Project_ID'])

# Expenditure might have multiple payments per project, so we group by Project_ID and sum the disbursed amount
# First, clean the amount column to be numeric
exp_df['Fund Disbursed Amount ( ₹ )'] = pd.to_numeric(exp_df['Fund Disbursed Amount ( ₹ )'].astype(str).str.replace(',', ''), errors='coerce')
exp_grouped = exp_df.groupby('Project_ID', as_index=False).agg({
    'Fund Disbursed Amount ( ₹ )': 'sum',
    'Payment Status': 'last'
})

comp_subset = comp_df[['Project_ID', 'Completion Date', 'Amount Disbursed ( ₹ )']].drop_duplicates(subset=['Project_ID'])

In [ ]:
# Merge datasets horizontally on Project_ID
# We use outer joins to ensure we don't lose any projects
master_df = rec_subset.merge(sanc_subset, on='Project_ID', how='outer')
master_df = master_df.merge(exp_grouped, on='Project_ID', how='outer')
master_df = master_df.merge(comp_subset, on='Project_ID', how='outer')

print(f"Master dataset created with {len(master_df)} rows and {len(master_df.columns)} columns.")

In [ ]:
# Show how beautifully the lifecycle tracks now!
master_df[['Project_ID', 'RECOMMENDED AMOUNT   ( ₹ )', 'Sanction Amount ( ₹ )', 'Fund Disbursed Amount ( ₹ )', 'Work Status']].head(10)

In [ ]:
# Save the merged master dataset for the next modules
master_df.to_csv('master_mplads_data.csv', index=False)
print("Saved to master_mplads_data.csv")